# 05 · Joint residual risk and diffusion clocks

**RQ2–RQ3:** Can explicit residual risk repair the smooth-factor model? Does
stochastic time scaling explain some calendar-gap behavior? This is a revised
development model, distinct from the frozen source-covariance benchmark.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from spde_covariance.empirical_research import load_research_evidence

evidence = load_research_evidence(Path("../experiments/results/shfe-research"))
print(evidence["protocol"]["status"])
print("Numerical source:", evidence["manifest"]["git_revision"])
print("Original baseline source:", evidence["protocol"]["baseline_source_revision"])
assert evidence["protocol"]["baseline_summary_sha256"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def table(rows, columns):
    print(" | ".join(columns))
    for row in rows:
        print(
            " | ".join(
                (
                    f"{row.get(c):.6g}" if isinstance(row.get(c), float) else str(row.get(c))
                    for c in columns
                )
            )
        )

## Model and estimand

For the current actual-contract set, collect historical adjacent changes and
their exact transport decomposition. For diffusion duration $h_i$, estimate

```math
z_i=\frac{1}{\sqrt{h_i}}\begin{pmatrix}r_i\\u_i\end{pmatrix},\qquad
\widehat Q=\frac1n\sum_i z_i z_i^\top,
\qquad \widehat K_{n+1}=h_{n+1}[E_{n+1},I]\widehat Q[E_{n+1},I]^\top.
```

This retains factor/residual cross blocks and is PSD by construction. Optional
diagonal shrinkage changes the joint moment explicitly. The locally common
joint moment is an approximation across contract ages, not physical forcing
identification. Compare with factor-only and raw direct second moments on the
**same complete historical intervals**.

Physical transport always uses ACT/365.25 maturity time. Only the diffusion
normalizer changes: $h_i=\Delta_i$ or one session/252. The original source-$C$
pipeline remains a separate calendar-time control.

In [ ]:
joint = evidence["joint"]
print(joint["clock_convention"])
print(joint["selection_criterion"])
print("Joint settings:", json.dumps(joint["joint_config"], indent=2))
for method, selection in joint["selection"].items():
    print(
        method,
        selection["selected"],
        "validation origins",
        selection["common_validation_origins"],
        "last outcome",
        selection["selection_last_outcome"],
    )
    assert selection["selection_last_outcome"] <= joint["chronology"]["validation_end"]

In [ ]:
table(
    joint["risk_summary"],
    [
        "exposure",
        "method",
        "common_count",
        "qlike",
        "realized_over_predicted_second_moment",
        "predicted_rms_cny",
        "realized_rms_cny",
    ],
)
table(
    [r for r in joint["paired_scores"] if r["score"] == "qlike"],
    [
        "exposure",
        "method",
        "reference",
        "count",
        "mean",
        "bootstrap_standard_error",
        "bootstrap_percentile_95",
    ],
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, exposure in zip(axes, ["outright_3m", "spread_3m_6m", "butterfly_3m_6m_9m"], strict=True):
    rows = [r for r in joint["risk_summary"] if r["exposure"] == exposure]
    ax.bar(
        range(len(rows)), [r["realized_over_predicted_second_moment"] for r in rows], color="0.45"
    )
    ax.axhline(1, color="black", linestyle=":")
    ax.set_xticks(range(len(rows)), [r["method"] for r in rows], rotation=80, fontsize=7)
    ax.set(title=exposure, ylabel="Realized / predicted second moment")
fig.tight_layout()
plt.show()

## Cross-covariances and numerical checks

Dropping cross terms would impose independence without evidence. The components
below sum to a joint-model forecast; factor/direct controls can use a different
kernel. Numerical PSD/rank diagnostics establish matrix feasibility, not a
statistical or market-model guarantee. Availability is reported separately.

In [ ]:
table(
    joint["risk_components"], ["method", "exposure", "count", "components_cny2", "cross_over_total"]
)
print(json.dumps(joint["numerical_diagnostics"], indent=2))
print(json.dumps(joint["availability"], indent=2))

## Interpretation and next decision

Compare the selected practical policies and the fixed-clock companions separately.
Residual risk and a different clock can improve a weak factor forecast without
establishing an SPDE advantage over raw direct risk. Use the matched attribution
results and effective sample counts to qualify the conclusion. Test the selected
revised policy on an uninspected period before promoting it beyond development.

## Reproduction and research boundary

This notebook reads checksummed aggregate evidence, not raw exchange files.
The numerical algorithms are in `spde_covariance`; use the
[protocol](../docs/empirical-research-protocol.md) to rerun local calibration.
Execute this notebook with `uv run python scripts/check_notebook.py notebooks/05_joint_residual_risk_and_clocks.ipynb`.

The original historical benchmark remains fixed. New analyses on the inspected
2024–2025 period are exploratory, even though forecast issuance is past-only.
No final untouched confirmation period, executable trading backtest or physical
forcing identification is claimed. See the [roadmap](../README.md).